In [1]:
print("hi")

hi


In [9]:
from transformers import Wav2Vec2FeatureExtractor
from transformers import AutoModel
import torch
import torch.nn.functional as F
import torchaudio.transforms as T
from datasets import Dataset as HFDataset, load_dataset
import os
from dotenv import load_dotenv
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
from torch.utils.data import Dataset as TorchDataset, DataLoader
import torchaudio.transforms as T
import librosa
import io
from datasets import Dataset as HFDataset, concatenate_datasets

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


In [3]:
def decode_audio(item):
    array, sr = librosa.load(io.BytesIO(item["audio"]["bytes"]), sr=None, mono=True)
    item["array"] = array
    item["sample_rate"] = sr
    return item

def collate_fn(batch, max_samples=24000*30):  # 30 seconds at 24kHz
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return torch.tensor([]), [], []

    audios, genres, indices = zip(*batch)
    # Truncate any clip longer than max_samples
    audios = [a[:max_samples] for a in audios]
    max_len = max(a.shape[0] for a in audios)
    padded = torch.stack([
        F.pad(a, (0, max_len - a.shape[0]))
        for a in audios
    ])
    return padded, list(genres), list(indices)

class FMAStreamDataset(TorchDataset):
    def __init__(self, hf_dataset, processor, window_size=150, sample_rate = 44100):
        self.dataset       = hf_dataset
        self.window_size   = window_size
        self.sample_rate   = sample_rate
        self.processor = processor
        self.resample_rate = processor.sampling_rate

    def __len__(self):
        return len(self.dataset)

    def __oldgetitem__(self, idx):
        item  = self.dataset[idx]
        item = decode_audio(item)
        audio = item["array"]
        return audio, item["genres"]

    def __getitem__(self, idx):
        item  = self.dataset[idx]
        item = decode_audio(item)
        audio = item["array"]

        sample_rate = item.get("sample_rate", self.sample_rate)
        try:
            if sample_rate != self.resample_rate:
                resampler  = T.Resample(sample_rate, self.resample_rate)
                audio_array = resampler(torch.tensor(audio).float())
            else:
                audio_array = torch.tensor(audio).float()
        except Exception as e:
            print(f"Failed clip {idx}: {e}")
            return None
        return audio_array, item["genres"], idx

    def __iter__(self):
        for idx, item in enumerate(self.dataset):
            audio = item["array"]
            sample_rate = item.get("sample_rate",self.sample_rate)
            try:
              if sample_rate != self.resample_rate:
                  resampler  = T.Resample(sample_rate, self.resample_rate)
                  audio_array = resampler(torch.tensor(audio).float())
              else:
                  audio_array = torch.tensor(audio).float()
              yield audio_array, item["genres"]

            except Exception as e:
                # skip corrupt/unreadable clips (173 were flagged in the dataset card)
                print(f"Skipping clip {idx}: {e}")
                continue


def load_fma_dataloaders(dataset, processor, streaming=False,window_size=150,
                         batch_size=32, token=None,
                         val_frac=0.1, test_frac=0.1, seed=42,
                         max_clips=None):
    if type(dataset) == str:
        raw = load_dataset(dataset, split="train", streaming=streaming, token=token,revision="main")
    else:
        raw = dataset

    # optionally cap the number of clips
    if max_clips is not None:
        raw = raw.take(max_clips)

    # materialize to map-style so we get a known length and random access
    dataset = HFDataset.from_generator(lambda: (item for item in raw))
    # dataset = dataset.map(decode_audio, remove_columns=["audio"])  # drop the raw bytes column to save RAM
    n_total  = len(dataset)
    n_test   = int(n_total * test_frac)
    n_val    = int(n_total * val_frac)
    n_train  = n_total - n_val - n_test

    print(f"Dataset size: {n_total} clips → train: {n_train}, val: {n_val}, test: {n_test}")

    splits = dataset.train_test_split(
        test_size=n_val + n_test,
        seed=seed,
    )
    val_test = splits["test"].train_test_split(
        test_size=n_test,
        seed=seed,
    )

    train_ds = FMAStreamDataset(splits["train"],  processor, window_size)
    val_ds   = FMAStreamDataset(val_test["train"], processor, window_size)
    test_ds  = FMAStreamDataset(val_test["test"], processor, window_size)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,collate_fn=collate_fn)
    valid_loader = DataLoader(val_ds,   batch_size=batch_size,collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,collate_fn=collate_fn)

    return train_loader, valid_loader, test_loader

In [4]:
class MERT_model():
  def __init__(self, mert_model, processor, window_size=150, deterministic = True, layer_idx=-1):
    self.mert_model = mert_model
    self.processor = processor
    self.layer_idx = layer_idx
    self.deterministic = deterministic
    self.window_size = window_size

  def get_hidden_states(self, audio_array):
        audio_array = audio_array.to(next(self.mert_model.parameters()).device)
        with torch.no_grad():
            outputs = self.mert_model(audio_array, output_hidden_states=True)

        hidden = outputs.hidden_states[self.layer_idx] # [T, 768]

        T = hidden.shape[1]
        if T >= self.window_size:
            if self.deterministic:
                hidden = hidden[:, :self.window_size, :]
            else:
                start = torch.randint(0, T - self.window_size + 1, (1,)).item()
                hidden = hidden[:, start:start + self.window_size, :]
        else:
            hidden = F.pad(hidden, (0, 0, 0, self.window_size - T))

        return hidden  # [batch, window_size, 768]

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mert_model = AutoModel.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True).to(device)
processor = Wav2Vec2FeatureExtractor.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True)
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# mert_model = AutoModel.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True).to(device)
# if torch.cuda.device_count() > 1:
#     mert_model = torch.nn.DataParallel(mert_model)
#     print(f"Using {torch.cuda.device_count()} GPUs")
# processor = Wav2Vec2FeatureExtractor.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True)
# print(f"Using device: {device}")

: 

In [6]:
import datasets
fma_dataset = load_dataset(
        "benjamin-paine/free-music-archive-medium",
        split="train",
        # streaming=True,
        token=HF_TOKEN,
        revision="main",
    ).cast_column("audio", datasets.Audio(decode=False))

In [7]:
#TODO Add L0 logging just cause

config={
        "layer_idx":   -1,
        "window_size": 1125, #~30 seconds consider cutting this in half
        "rho":         0.05,
        "lr":          1e-4,
        "batch_size":  8,
        "rho_loss_weight": .5, #Right down the middle. Good performance in the original paper
        "epochs": 1
    }

# "train_loader, valid_loader, test_loader = load_fma_dataloaders(
#     fma_dataset, processor,
#     window_size=config["window_size"],
#     batch_size=config["batch_size"],
#     token=HF_TOKEN,
# )
MERT = MERT_model(mert_model, processor, deterministic = True,
                  window_size=config["window_size"], layer_idx=config["layer_idx"])
rho = config["rho"]
epochs = config["epochs"]

In [ ]:



layer_idx = config["layer_idx"]
window_size = config["window_size"]
batch_size = config["batch_size"]

input_dir = f"orcd/scratch"
output_dir = f"mert_medium/window{window_size}_layer{layer_idx}"
checkpoint_dir = os.path.join(input_dir, "checkpoints")

os.makedirs(checkpoint_dir, exist_ok=True)

CHECKPOINT_EVERY = 50

KEEP_COLS = [
    "title", "url", "artist", "composer", "lyricist", "publisher",
    "genres", "tags", "released", "language", "listens", "artist_url",
    "artist_website", "album_title", "album_url", "license", "copyright",
    "explicit", "instrumental", "allow_commercial_use", "allow_derivatives",
    "require_attribution", "require_share_alike",
]

full_ds = FMAStreamDataset(fma_dataset, processor, window_size)
full_loader = DataLoader(full_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

available_cols = [c for c in KEEP_COLS if c in fma_dataset.column_names]
metadata_table = fma_dataset.select_columns(available_cols)

# Resume support
existing_checkpoints = sorted([
    f for f in os.listdir(checkpoint_dir)
    if f.startswith("ckpt_") and f.endswith(".parquet")
]) if os.path.exists(checkpoint_dir) else []
start_batch = len(existing_checkpoints) * CHECKPOINT_EVERY
checkpoint_idx = len(existing_checkpoints)

if start_batch > 0:
    print(f"Found {len(existing_checkpoints)} existing checkpoints, resuming from batch {start_batch}")

mert_outputs = []
metadata_rows = []
batch_count = 0

for batch_idx, (audio_array, genres, indices) in enumerate(full_loader):
    if batch_idx < start_batch:
        continue

    if audio_array.numel() == 0:
        continue

    with torch.no_grad():
        hidden = MERT.get_hidden_states(audio_array)

    for i in range(hidden.shape[0]):
        mert_outputs.append(hidden[i].cpu().numpy().tobytes())
        metadata_rows.append(metadata_table[indices[i]])

    batch_count += 1

    if batch_count % 10 == 0:
        print(f"Processed batch {batch_count}, total samples: {len(mert_outputs)}")

    if batch_count % CHECKPOINT_EVERY == 0:
        ckpt_dict = {"MERT_output": mert_outputs}
        for col in available_cols:
            ckpt_dict[col] = [r[col] for r in metadata_rows]
        ckpt_ds = HFDataset.from_dict(ckpt_dict)
        ckpt_path = os.path.join(checkpoint_dir, f"ckpt_{checkpoint_idx:04d}.parquet")
        ckpt_ds.to_parquet(ckpt_path)
        print(f"Saved checkpoint {checkpoint_idx} ({len(mert_outputs)} samples) -> {ckpt_path}")
        mert_outputs = []
        metadata_rows = []
        checkpoint_idx += 1

if mert_outputs:
    ckpt_dict = {"MERT_output": mert_outputs}
    for col in available_cols:
        ckpt_dict[col] = [r[col] for r in metadata_rows]
    ckpt_ds = HFDataset.from_dict(ckpt_dict)
    ckpt_path = os.path.join(checkpoint_dir, f"ckpt_{checkpoint_idx:04d}.parquet")
    ckpt_ds.to_parquet(ckpt_path)
    print(f"Saved final checkpoint {checkpoint_idx} ({len(mert_outputs)} samples)")

print("All batches processed.")

Found 62 existing checkpoints, resuming from batch 3100


[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (3264) too large for available bit count (3224)
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!


In [11]:
import glob
import shutil

layer_idx = config["layer_idx"]
window_size = config["window_size"]
input_dir = f"orcd/scratch"
output_dir = f"orcd/scratch/window{window_size}_layer{layer_idx}"
checkpoint_dir = os.path.join(input_dir, "checkpoints")

parquet_files = sorted(glob.glob(os.path.join(checkpoint_dir, "ckpt_*.parquet")))
partial_datasets = [HFDataset.from_parquet(f) for f in parquet_files]
final_dataset = concatenate_datasets(partial_datasets)

# Deduplicate by URL
if "url" in final_dataset.column_names:
    urls_seen = set()
    unique_indices = []
    for i, url in enumerate(final_dataset["url"]):
        if url not in urls_seen:
            urls_seen.add(url)
            unique_indices.append(i)
    before = len(final_dataset)
    final_dataset = final_dataset.select(unique_indices)
    after = len(final_dataset)
    if before != after:
        print(f"Removed {before - after} duplicates")

final_path = os.path.join(output_dir, "dataset")
os.makedirs(final_path, exist_ok=True)
final_dataset.save_to_disk(final_path)
print(f"Saved {len(final_dataset)} samples -> {final_path}")
print(f"Columns: {final_dataset.column_names}")

shutil.rmtree(checkpoint_dir)
print("Checkpoints deleted.")

Saving the dataset (172/172 shards): 100%|██████████| 24801/24801 [03:59<00:00, 103.45 examples/s]


Saved 24801 samples -> orcd/scratch/window1125_layer-1/dataset
Columns: ['MERT_output', 'title', 'url', 'artist', 'composer', 'lyricist', 'publisher', 'genres', 'tags', 'released', 'language', 'listens', 'artist_url', 'artist_website', 'album_title', 'album_url', 'license', 'copyright', 'explicit', 'instrumental', 'allow_commercial_use', 'allow_derivatives', 'require_attribution', 'require_share_alike']
Checkpoints deleted.


In [ ]:
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")

GPU memory allocated: 0.00 GB
GPU memory reserved: 0.00 GB


: 

In [ ]:
print(next(mert_model.parameters()).device)

cuda:0
